# CNN + GLCM baselines

Trains and evaluates ten hybrid classifiers that combine an ImageNet-pretrained CNN backbone (ResNet, EfficientNet, MobileNet, DenseNet, VGG) with 18 GLCM texture features, on the same GT-cropped and preprocessed (bilateral filter + CLAHE) bean images used by the proposed system. The models to run are chosen interactively (default: all).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Hybrid pretrained CNN + GLCM baselines for cocoa bean classification.
#
# Input: ground-truth (GT) cropped bean images from the main training pipeline
# (cropped_splits_gt_based_backup/), not full images, so that the baselines are
# evaluated under the same conditions as the proposed system
# (RetinaNet-DE + AdaptiveSwin-CNN + XGBoost), whose classifier also takes GT crops.
#
# Preprocessing matches the main pipeline:
#   - Bilateral filter (d=5, sigmaColor=50, sigmaSpace=50)
#   - CLAHE (clipLimit=1.5, tileGridSize=(4,4))

# ===== 1. IMPORT LIBRARIES =====
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_recall_fscore_support)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import cv2
from skimage.feature import graycomatrix, graycoprops
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import (
    ResNet50, ResNet101,
    EfficientNetB0, EfficientNetB3,
    MobileNetV2, MobileNetV3Large,
    DenseNet121, DenseNet169,
    VGG16, VGG19
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                        ModelCheckpoint)
import json, time, pickle, warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

np.random.seed(42)
tf.random.set_seed(42)

# Matplotlib publication style
matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'font.size':         12,
    'axes.titlesize':    13,
    'axes.labelsize':    12,
    'xtick.labelsize':   11,
    'ytick.labelsize':   11,
    'legend.fontsize':   11,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

print("="*90)
print(" "*15 + "HYBRID CNN + GLCM — COMPARATIVE BASELINE STUDY")
print(" "*10 + "Input: GT-Cropped Images (Aligned with Proposed System)")
print("="*90)
print(f"TensorFlow: {tf.__version__} | GPU: {len(tf.config.list_physical_devices('GPU'))} device(s)")
print("="*90)

# ===== 2. CONFIGURATION =====
# GT-cropped splits from the main training pipeline.
# Layout: {split}/{class_name}/crop_*.jpg
BASE_DRIVE_DIR = '/content/drive/MyDrive/UMK/Datasets/MCC-02-dataset'

SPLIT_DIR   = os.path.join(BASE_DRIVE_DIR, 'cropped_splits_gt_based_backup')
MODEL_DIR   = '/content/drive/MyDrive/UMK/Models_GTCrop'
RESULTS_DIR = '/content/drive/MyDrive/UMK/Results_CNN_GLCM_GTCrop'

for d in [MODEL_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE    = 224
BATCH_SIZE  = 64
EPOCHS      = 30
CLASSES     = ['Broken', 'Unfermented', 'Fermented', 'Moldy']
NUM_CLASSES = len(CLASSES)

# GLCM configuration
GLCM_DISTANCES  = [1, 3, 5]
GLCM_ANGLES     = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GLCM_PROPERTIES = ['contrast', 'dissimilarity', 'homogeneity',
                   'energy', 'correlation', 'ASM']
GLCM_DIM = len(GLCM_DISTANCES) * len(GLCM_PROPERTIES)   # = 18

AVAILABLE_MODELS = {
    '1':  {'name': 'ResNet50',       'class': ResNet50,        'selected': False},
    '2':  {'name': 'ResNet101',      'class': ResNet101,       'selected': False},
    '3':  {'name': 'EfficientNetB0', 'class': EfficientNetB0,  'selected': False},
    '4':  {'name': 'EfficientNetB3', 'class': EfficientNetB3,  'selected': False},
    '5':  {'name': 'MobileNetV2',    'class': MobileNetV2,     'selected': False},
    '6':  {'name': 'MobileNetV3Large','class': MobileNetV3Large,'selected': False},
    '7':  {'name': 'DenseNet121',    'class': DenseNet121,     'selected': False},
    '8':  {'name': 'DenseNet169',    'class': DenseNet169,     'selected': False},
    '9':  {'name': 'VGG16',          'class': VGG16,           'selected': False},
    '10': {'name': 'VGG19',          'class': VGG19,           'selected': False},
}

# ===== 3. MODEL SELECTION (JS prompt, 10 s timeout) =====
from google.colab import output

def select_models():
    print("\n" + "="*90)
    print("MODEL SELECTION")
    print("="*90)
    print("\nAvailable Models:")
    for key, val in AVAILABLE_MODELS.items():
        print(f"  [{key:>2}] {val['name']}")
    print("\n  Enter model numbers (e.g., 1,3,5), range (1-5), or 'all'")
    print("  Auto-selects 'all' after 10 seconds\n")

    js_code = """
    new Promise((resolve) => {
        const div = document.createElement('div');
        div.innerHTML = `
            <div style="margin:10px 0;padding:12px;border:1px dashed #777;border-radius:5px;
                        background:#f8f9fa;font-family:monospace;">
                <p id="timer_msg" style="color:#d9534f;font-weight:bold;margin-bottom:8px;">
                    ⏳ Auto-selecting 'all' in: 10s</p>
                <label style="font-weight:bold;">Selection: </label>
                <input type="text" id="sel_input" placeholder="e.g. 1,3 or all"
                       style="padding:4px 8px;width:160px;border:1px solid #ccc;
                              border-radius:3px;margin:0 6px;">
                <button id="sel_btn"
                        style="padding:4px 14px;background:#4CAF50;color:#fff;
                               border:none;border-radius:3px;cursor:pointer;">
                    Confirm
                </button>
            </div>`;
        document.body.appendChild(div);

        const inp = div.querySelector('#sel_input');
        const btn = div.querySelector('#sel_btn');
        const tmr = div.querySelector('#timer_msg');
        setTimeout(() => inp.focus(), 100);

        let t = 10, done = false;
        const iv = setInterval(() => {
            t--;
            if (t <= 0) {
                clearInterval(iv);
                if (!done) {
                    tmr.innerText = "⏳ Timeout — using 'all'";
                    inp.disabled = btn.disabled = true;
                    setTimeout(() => { div.remove(); resolve('all'); }, 1200);
                }
            } else { tmr.innerText = `⏳ Auto-selecting 'all' in: ${t}s`; }
        }, 1000);

        btn.onclick = () => {
            done = true; clearInterval(iv);
            const v = inp.value.trim().toLowerCase() || 'all';
            div.remove(); resolve(v);
        };
        inp.onkeydown = (e) => { if (e.key === 'Enter') btn.click(); };
    });
    """

    try:
        selection = output.eval_js(js_code)
    except Exception as e:
        print(f"Interactive UI failed: {e} — using 'all'")
        selection = 'all'

    print(f"\nSelection received: '{selection}'")
    selected = []

    if selection in ('all', ''):
        selected = list(AVAILABLE_MODELS.keys())
    else:
        for part in selection.replace(' ', '').split(','):
            if '-' in part:
                lo, hi = part.split('-')
                lo = lo.lstrip('a')
                hi = hi.lstrip('a')
                try:
                    for n in range(int(lo), int(hi)+1):
                        if str(n) in AVAILABLE_MODELS:
                            selected.append(str(n))
                except ValueError:
                    pass
            elif part in AVAILABLE_MODELS:
                selected.append(part)
            else:
                print(f"  Invalid key ignored: {part}")

    if not selected:
        print("  No valid selection — using 'all'")
        selected = list(AVAILABLE_MODELS.keys())

    print(f"\n  Running {len(selected)} model(s):")
    for k in selected:
        AVAILABLE_MODELS[k]['selected'] = True
        print(f"    - {AVAILABLE_MODELS[k]['name']}")

    return selected

# ===== 4. PREPROCESSING (same as the main training pipeline) =====
def apply_bilateral_clahe(img_rgb_uint8):
    """Bilateral filter (d=5, sigmaColor=50, sigmaSpace=50) followed by
    CLAHE on the L channel (clipLimit=1.5, tileGridSize=(4,4))."""
    img_filtered = cv2.bilateralFilter(
        img_rgb_uint8, d=5, sigmaColor=50, sigmaSpace=50)
    lab = cv2.cvtColor(img_filtered, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4, 4)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

# ===== 5. GLCM EXTRACTION =====
def extract_glcm_features(img_rgb_uint8):
    """Extract 18-dim GLCM texture features (3 distances × 6 properties)."""
    gray = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2GRAY)
    feats = []
    for dist in GLCM_DISTANCES:
        glcm = graycomatrix(gray, distances=[dist], angles=GLCM_ANGLES,
                            levels=256, symmetric=True, normed=True)
        for prop in GLCM_PROPERTIES:
            feats.append(float(np.mean(graycoprops(glcm, prop))))
    return np.array(feats, dtype=np.float32)

# ===== 6. DATASET LOADER — GT-Cropped Images =====
def load_split_data(split_dir, split_name):
    """
    Load GT-cropped images from {split_dir}/{class_name}/crop_*.jpg.
    No center crop is applied because each image is already a tight bean crop.
    """
    images, glcm_feats, labels = [], [], []
    filenames, folders, full_paths = [], [], []

    print(f"\n{'='*90}")
    print(f"LOADING {split_name} — GT-Cropped Images")
    print(f"Path: {split_dir}")
    print(f"{'='*90}")

    for cls_idx, cls_name in enumerate(CLASSES):
        cls_path = os.path.join(split_dir, cls_name)
        if not os.path.exists(cls_path):
            print(f"  Not found: {cls_path}")
            continue

        img_files = []
        for root, _, files in os.walk(cls_path):
            for f in files:
                if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif','.tiff')):
                    img_files.append(os.path.join(root, f))

        print(f"\n  [{cls_idx+1}/{NUM_CLASSES}] {cls_name}: {len(img_files)} GT crops")
        ok = 0; err = 0

        for img_path in img_files:
            try:
                img_bgr = cv2.imread(img_path)
                if img_bgr is None:
                    err += 1; continue

                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

                # Images are already GT bounding-box crops; resize directly
                img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE),
                                         interpolation=cv2.INTER_LINEAR)

                # Preprocessing: bilateral filter + CLAHE
                img_processed = apply_bilateral_clahe(img_resized)

                # GLCM features from the preprocessed image
                glcm = extract_glcm_features(img_processed)

                # Scale to [0, 1] for the CNN
                img_norm = img_processed.astype(np.float32) / 255.0

                images.append(img_norm)
                glcm_feats.append(glcm)
                labels.append(cls_idx)
                filenames.append(os.path.basename(img_path))
                folders.append(cls_name)
                full_paths.append(img_path)
                ok += 1

            except Exception as e:
                err += 1; continue

        print(f"    Loaded: {ok} | Errors: {err}")

    return (np.array(images, dtype=np.float32),
            np.array(glcm_feats, dtype=np.float32),
            np.array(labels, dtype=np.int32),
            filenames, folders, full_paths)

# ===== 7. DATA PREPARATION =====
print("\nLoading GT-cropped dataset splits...")

X_train_img, X_train_glcm, y_train, train_files, train_folders, train_paths = \
    load_split_data(os.path.join(SPLIT_DIR,'train'), "TRAIN")
X_val_img,   X_val_glcm,   y_val,   val_files,   val_folders,   val_paths   = \
    load_split_data(os.path.join(SPLIT_DIR,'val'),   "VALIDATION")
X_test_img,  X_test_glcm,  y_test,  test_files,  test_folders,  test_paths  = \
    load_split_data(os.path.join(SPLIT_DIR,'test'),  "TEST")

total = len(X_train_img) + len(X_val_img) + len(X_test_img)

print(f"\n{'='*90}")
print("DATASET SUMMARY")
print("="*90)
print(f"  Input type    : GT-Cropped images (same as proposed system)")
print(f"  Preprocessing : Bilateral Filter + CLAHE (aligned with main system)")
print(f"  Total samples : {total}")
print(f"  Train : {len(X_train_img):5d} ({len(X_train_img)/total*100:.1f}%)")
print(f"  Val   : {len(X_val_img):5d} ({len(X_val_img)/total*100:.1f}%)")
print(f"  Test  : {len(X_test_img):5d} ({len(X_test_img)/total*100:.1f}%)")
print(f"  GLCM feature dim: {X_train_glcm.shape[1]}")

# Normalize GLCM features
scaler = StandardScaler()
X_train_glcm_sc = scaler.fit_transform(X_train_glcm)
X_val_glcm_sc   = scaler.transform(X_val_glcm)
X_test_glcm_sc  = scaler.transform(X_test_glcm)

with open(os.path.join(MODEL_DIR, 'glcm_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

class_weights_arr  = compute_class_weight('balanced',
                       classes=np.unique(y_train), y=y_train)
class_weight_dict  = dict(enumerate(class_weights_arr))

print(f"\nClass weights (from training data):")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:<14}: {class_weights_arr[i]:.4f}")
print("="*90)

# ===== 8. MODEL BUILDER =====
def create_hybrid_model(model_name, model_class,
                        img_shape=(224,224,3), glcm_dim=18, n_classes=4):
    """
    Hybrid CNN + GLCM model.
    CNN branch : pretrained backbone (frozen) → GAP → Dense
    GLCM branch: Dense layers on 18-dim texture features
    Fusion     : Concatenate → Dense → Softmax
    """
    # CNN branch
    img_in    = layers.Input(shape=img_shape, name='image_input')
    base      = model_class(include_top=False, weights='imagenet',
                            input_tensor=img_in, pooling=None)
    base.trainable = False

    x = layers.GlobalAveragePooling2D(name='cnn_gap')(base.output)
    x = layers.BatchNormalization(name='cnn_bn1')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu', name='cnn_fc1')(x)
    x = layers.BatchNormalization(name='cnn_bn2')(x)
    x = layers.Dropout(0.4)(x)
    cnn_feat  = layers.Dense(256, activation='relu', name='cnn_features')(x)

    # GLCM branch
    glcm_in   = layers.Input(shape=(glcm_dim,), name='glcm_input')
    g = layers.Dense(128, activation='relu', name='glcm_fc1')(glcm_in)
    g = layers.BatchNormalization(name='glcm_bn1')(g)
    g = layers.Dropout(0.4)(g)
    g = layers.Dense(64, activation='relu', name='glcm_fc2')(g)
    g = layers.BatchNormalization(name='glcm_bn2')(g)
    g = layers.Dropout(0.3)(g)
    glcm_feat = layers.Dense(32, activation='relu', name='glcm_features')(g)

    # Fusion
    z = layers.Concatenate(name='fusion')([cnn_feat, glcm_feat])
    z = layers.Dense(256, activation='relu', name='fusion_fc1')(z)
    z = layers.BatchNormalization(name='fusion_bn')(z)
    z = layers.Dropout(0.5)(z)
    z = layers.Dense(128, activation='relu', name='fusion_fc2')(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(n_classes, activation='softmax', name='output')(z)

    model = models.Model(inputs=[img_in, glcm_in], outputs=out,
                         name=f'{model_name}_GLCM')
    return model, base

# ===== 9. TRAINING FUNCTION =====
def train_model(model_name, model_class):
    print(f"\n{'─'*60}")
    print(f"TRAINING: {model_name} + GLCM")
    print(f"{'─'*60}")

    model, _ = create_hybrid_model(
        model_name, model_class,
        img_shape=(IMG_SIZE, IMG_SIZE, 3),
        glcm_dim=X_train_glcm_sc.shape[1],
        n_classes=NUM_CLASSES)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'])

    ckpt_path = os.path.join(MODEL_DIR, f'hybrid_{model_name}_gtcrop_best.keras')
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2,
                          patience=4, min_lr=1e-6, verbose=1),
        ModelCheckpoint(ckpt_path, monitor='val_loss',
                        save_best_only=True, verbose=1),
    ]

    t0 = time.time()
    history = model.fit(
        x=[X_train_img, X_train_glcm_sc],
        y=y_train_cat,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=([X_val_img, X_val_glcm_sc], y_val_cat),
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1)
    train_time = time.time() - t0

    with open(os.path.join(RESULTS_DIR, f'history_{model_name}.pkl'), 'wb') as f:
        pickle.dump(history.history, f)

    print(f"\n  Training complete: {train_time:.1f}s ({train_time/60:.2f} min)")
    return model, history, train_time

# ===== 10. EVALUATION FUNCTION =====
def evaluate_model(model_name):
    print(f"\n{'='*90}")
    print(f"EVALUATING: {model_name} + GLCM (GT-Cropped Test Set)")
    print("="*90)

    ckpt_path = os.path.join(MODEL_DIR, f'hybrid_{model_name}_gtcrop_best.keras')
    model = keras.models.load_model(ckpt_path)

    y_pred_prob = model.predict(
        [X_test_img, X_test_glcm_sc], batch_size=BATCH_SIZE, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_test_cat,  axis=1)

    print("\nClassification Report:")
    print("─"*60)
    report = classification_report(y_true, y_pred,
                                   target_names=CLASSES, digits=4)
    print(report)

    acc           = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    prec_pc, rec_pc, f1_pc, supp_pc = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0)

    mis_idx = np.where(y_pred != y_true)[0]
    mis_data = [{
        'filename':   test_files[i],
        'folder':     test_folders[i],
        'true_label': CLASSES[y_true[i]],
        'pred_label': CLASSES[y_pred[i]],
        'path':       test_paths[i],
    } for i in mis_idx]
    mis_df = pd.DataFrame(mis_data)

    print(f"  Total misclassified: {len(mis_idx)} / {len(y_true)}")

    return {
        'model_name':          model_name,
        'accuracy':            acc,
        'precision':           prec,
        'recall':              rec,
        'f1':                  f1,
        'precision_per_class': prec_pc,
        'recall_per_class':    rec_pc,
        'f1_per_class':        f1_pc,
        'support_per_class':   supp_pc,
        'y_true':              y_true,
        'y_pred':              y_pred,
        'y_pred_probs':        y_pred_prob,
        'misclassified_df':    mis_df,
        'misclassified_indices': mis_idx,
        'confusion_matrix':    confusion_matrix(y_true, y_pred),
        'classification_report': report,
    }

# ===== 11. PLOT: TRAINING HISTORY =====
def plot_training_history(history, model_name):
    hist = history.history if hasattr(history, 'history') else history
    epochs = range(1, len(hist['accuracy']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} + GLCM — Training History',
                 fontweight='bold', fontsize=13)

    ax1.plot(epochs, hist['accuracy'],     'b-', lw=2, label='Training Accuracy')
    ax1.plot(epochs, hist['val_accuracy'], 'r-', lw=2, label='Validation Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.set_title('Accuracy per Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, hist['loss'],     'b-', lw=2, label='Training Loss')
    ax2.plot(epochs, hist['val_loss'], 'r-', lw=2, label='Validation Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.set_title('Loss per Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'{model_name}_training_history.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# ===== 12. PLOT: CONFUSION MATRIX =====
def plot_confusion_matrix(results):
    mn = results['model_name']
    cm = results['confusion_matrix']
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{mn} + GLCM — Confusion Matrix (GT-Cropped Test Set)',
                 fontweight='bold', fontsize=13)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                ax=ax1, linewidths=0.5)
    ax1.set_title('Count');  ax1.set_ylabel('True Label')
    ax1.set_xlabel('Predicted Label')

    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens',
                xticklabels=CLASSES, yticklabels=CLASSES,
                ax=ax2, linewidths=0.5)
    ax2.set_title('Percentage (%)'); ax2.set_ylabel('True Label')
    ax2.set_xlabel('Predicted Label')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'{mn}_confusion_matrix.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# ===== 13. PLOT: MISCLASSIFIED SAMPLES =====
def plot_misclassified_samples(results):
    mn      = results['model_name']
    mis_df  = results['misclassified_df']
    mis_idx = results['misclassified_indices']

    if len(mis_df) == 0:
        print(f"  No misclassified samples for {mn}")
        return

    n = min(8, len(mis_df))
    fig = plt.figure(figsize=(20, 5 * ((n + 3) // 4)))
    fig.suptitle(f'{mn} + GLCM — Misclassified Samples (GT-Cropped)',
                 fontsize=13, fontweight='bold')

    for i in range(n):
        idx = mis_idx[i]
        img = cv2.cvtColor(cv2.imread(test_paths[idx]), cv2.COLOR_BGR2RGB)
        ax  = plt.subplot((n + 3) // 4, 4, i + 1)
        ax.imshow(img); ax.axis('off')
        ax.set_title(
            f"{test_files[idx]}\n"
            f"True: {test_folders[idx]}  |  Pred: {CLASSES[results['y_pred'][idx]]}",
            fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'{mn}_misclassified.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

# ===== 14. COMPARE ALL MODELS =====
def compare_models(all_results):
    print(f"\n{'='*90}")
    print(" "*25 + "COMPARATIVE BASELINE RESULTS")
    print("="*90)
    print("  Input: GT-Cropped images (Bilateral Filter + CLAHE)")
    print("  Evaluation protocol: identical to proposed system (A8)")
    print("="*90)

    rows = []
    for r in all_results:
        rows.append({
            'Model':          f"{r['model_name']} + GLCM",
            'Accuracy (%)':   f"{r['accuracy']*100:.2f}",
            'Precision (%)':  f"{r['precision']*100:.2f}",
            'Recall (%)':     f"{r['recall']*100:.2f}",
            'F1-Score (%)':   f"{r['f1']*100:.2f}",
            'Misclassified':  f"{len(r['misclassified_df'])}/{len(y_test)}",
        })

    df = pd.DataFrame(rows)
    print(f"\n{df.to_string(index=False)}\n")
    df.to_csv(os.path.join(RESULTS_DIR, 'baseline_comparison.csv'), index=False)
    print(f"  Saved: {RESULTS_DIR}/baseline_comparison.csv")

    # Metric comparison bar chart
    metrics_map = [('accuracy','Accuracy'),('precision','Precision'),
                   ('recall','Recall'),('f1','F1-Score')]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.ravel()
    fig.suptitle('Hybrid CNN + GLCM Baseline — Performance Comparison\n'
                 '(GT-Cropped Input, Aligned with Proposed System)',
                 fontsize=13, fontweight='bold')

    colors = plt.cm.tab10(np.linspace(0, 1, len(all_results)))
    for idx, (metric, label) in enumerate(metrics_map):
        ax = axes[idx]
        mnames = [r['model_name'] for r in all_results]
        vals   = [r[metric]*100   for r in all_results]
        bars = ax.bar(mnames, vals, color=colors, edgecolor='black',
                      linewidth=1.0, alpha=0.85)
        ax.set_title(label, fontweight='bold')
        ax.set_ylabel('Score (%)'); ax.set_ylim(0, 108)
        ax.grid(True, alpha=0.3, axis='y')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                    f'{val:.2f}%', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=40, ha='right')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'baseline_comparison_chart.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Final comparison against the proposed system
    print(f"\n{'='*90}")
    print("  Comparison with Proposed System")
    print("="*90)
    print(f"  {'Method':<42} {'Input':<10} {'Acc(%)':<10} {'F1(%)':<10}")
    print("  " + "─"*74)
    for r in all_results:
        print(f"  {r['model_name']+' + GLCM':<42} {'Cropped':<10} "
              f"{r['accuracy']*100:<10.2f} {r['f1']*100:<10.2f}")
    print("  " + "─"*74)
    print(f"  {'AdaptiveSwin-CNN + XGBoost [Ours]':<42} {'Cropped':<10} "
          f"{'99.11':<10} {'99.15':<10}  <- proposed system")
    print("="*90)
    print("  Note: All methods evaluated on identical GT-cropped test set")
    print("  using same preprocessing (Bilateral Filter + CLAHE, 224×224).")
    return df

# ===== 15. MAIN =====
def main():
    selected_keys = select_models()
    if not selected_keys:
        print("Exiting."); return None, None

    all_results = {}

    for i, key in enumerate(selected_keys):
        info = AVAILABLE_MODELS[key]
        mname, mclass = info['name'], info['class']

        print(f"\n{'='*90}")
        print(f"  MODEL {i+1}/{len(selected_keys)}: {mname}")
        print(f"{'='*90}")

        try:
            model, history, t = train_model(mname, mclass)
            results = evaluate_model(mname)
            results['training_time'] = t
            results['history']       = history

            plot_training_history(history, mname)
            plot_confusion_matrix(results)
            plot_misclassified_samples(results)
            results['misclassified_df'].to_csv(
                os.path.join(RESULTS_DIR, f'{mname}_misclassified.csv'), index=False)

            all_results[mname] = results

            print(f"\n  {mname} done | "
                  f"Acc: {results['accuracy']*100:.2f}% | "
                  f"Time: {t/60:.2f} min")

        except Exception as e:
            print(f"\n  ERROR {mname}: {e}")
            import traceback; traceback.print_exc()

    if all_results:
        comp_df = compare_models(list(all_results.values()))

        # Save full summary
        summary_lines = [
            "="*90,
            "HYBRID CNN + GLCM — COMPARATIVE BASELINE STUDY",
            'Input: ground-truth crops (same input as the proposed system)',
            "Preprocessing: Bilateral Filter + CLAHE (clipLimit=1.5, tile=4x4)",
            "="*90,
            f"Total images: {total}",
            f"GLCM features: {X_train_glcm_sc.shape[1]}",
            "",
        ]
        for r in all_results.values():
            summary_lines += [
                f"{r['model_name']} + GLCM:",
                f"  Accuracy : {r['accuracy']*100:.2f}%",
                f"  Precision: {r['precision']*100:.2f}%",
                f"  Recall   : {r['recall']*100:.2f}%",
                f"  F1-Score : {r['f1']*100:.2f}%",
                f"  Training : {r['training_time']/60:.2f} min",
                f"  Misclassified: {len(r['misclassified_df'])}/{len(y_test)}",
                "",
            ]
        summary_lines += [
            "="*90,
            "Note: Evaluation protocol identical to proposed system.",
            "All results on GT-cropped test set with same preprocessing.",
            "="*90,
        ]
        summary_text = "\n".join(summary_lines)
        with open(os.path.join(RESULTS_DIR, 'baseline_summary.txt'), 'w') as f:
            f.write(summary_text)
        print(f"\n{summary_text}")
        return list(all_results.values()), comp_df
    else:
        print("No models successfully trained.")
        return None, None

# ===== 16. RUN =====
if __name__ == "__main__":
    results, comparison = main()
    if results:
        print(f"\n  Process complete — {len(results)} model(s) trained")
        print(f"  Outputs: {RESULTS_DIR}/")